In [1]:
import mujoco
import numpy as np

model = mujoco.MjModel.from_xml_path("/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/efficold/efficold.xml")
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

trajectory = []

# ID
site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'target_maniglia')
joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'giunto_porta')
qpos_adr = model.jnt_qposadr[joint_id]

# Joint Coordinates
joint_center = data.xanchor[joint_id].copy()
Xj, Yj, Zj = joint_center[0], joint_center[1], joint_center[2]

pos_handle = data.site_xpos[site_id].copy()
Xh, Yh, Zh = pos_handle[0], pos_handle[1], pos_handle[2]

# calculate radius
radius = np.sqrt((Yh - Yj)**2 + (Zh - Zj)**2)

# save points
parameters = np.array([[Xh, Yj, Zj, radius]])
np.savetxt(f"/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/efficold/traiettorie/parametri_cerchio.csv", 
               parameters, delimiter=",", header="Centro_X,Centro_Y,Centro_Z,Raggio", comments='')

# Simulations
steps = 1000
target_angles = np.linspace(0, -2.0944, steps)

mujoco.mj_resetData(model, data)

for angle in target_angles:
    data.qpos[qpos_adr] = angle 
    mujoco.mj_forward(model, data) 
    
    pos = data.site_xpos[site_id].copy()
    
    quat = np.zeros(4)
    mujoco.mju_mat2Quat(quat, data.site_xmat[site_id])
    

    trajectory.append(np.concatenate([pos, quat]))

# save
np.save('traiettoria_porta.npy', np.array(trajectory))
np.savetxt("/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/efficold/traiettorie/traiettoria_porta.csv", trajectory, delimiter=",")
print(f"Traiettoria di {len(trajectory)} punti salvata!")

Traiettoria di 1000 punti salvata!
